In [ ]:
# Import shared loading/cleaning functions from utils.py
from utils import load_clean_data
from utils import get_cv_folds

import pandas as pd
import numpy as np

# Models we will train and compare
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

# For scaling numeric features (needed for Linear Regression and KNN)
from sklearn.preprocessing import StandardScaler

# For one-hot encoding the categorical columns
from sklearn.preprocessing import OneHotEncoder

# For evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [7]:
# Load the cleaned dataset
df = load_clean_data("data.csv")

# Generate the FIXED set of cross-validation folds
# Using the same random_state (set inside get_cv_folds) ensures these folds
# are identical to the folds used in every other encoder notebook
folds = get_cv_folds(df, n_splits=10)

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Generated {len(folds)} folds")

Loaded 79159 rows, 17 columns
Generated 10 folds


In [8]:
# The target variable we're predicting
y = df["advertised_price"]

# Drop the target from the feature set, everything else is a potential predictor
X = df.drop(columns=["advertised_price"])

# Identify which columns are categorical (need encoding) vs numeric (used as-is)
categorical_cols = ["make", "model", "generation", "fuel", "body",
                     "transmission", "trim", "plate_age_id"]

numeric_cols = ["mileage", "list_price", "reg_year", "doors", "engine",
                "engine_missing", "vehicle_age_days", "vehicle_age_years"]

# Sanity check: these two lists together should account for every column in X
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Total columns in X: {X.shape[1]}")
print(f"Accounted for: {len(categorical_cols) + len(numeric_cols)}")

Categorical columns: 8
Numeric columns: 8
Total columns in X: 16
Accounted for: 16


In [20]:
# Reset results dictionary, storing RMSE, MAE, and R² for each model
results = {
    "Linear Regression": {"rmse": [], "mae": [], "r2": []},
    "KNN": {"rmse": [], "mae": [], "r2": []},
    "Decision Tree": {"rmse": [], "mae": [], "r2": []},
    "Random Forest": {"rmse": [], "mae": [], "r2": []},
    "Stacking": {"rmse": [], "mae": [], "r2": []}
}

In [21]:
# Loop through each of the 10 fixed folds
# fold_num just labels which fold we're on (1 to 5), for printing progress
for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):

    # ---- Split data into this fold's train/test portions ----
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ---- One-Hot Encode the categorical columns ----
    # handle_unknown="ignore" means if a category appears in the test fold
    # that was never seen during training, it's encoded as all zeros instead
    # of throwing an error (this can happen with rare categories, e.g. "trim")
    # sparse_output=False returns a regular dense array, easier to work with
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    # IMPORTANT: fit the encoder ONLY on the training data for this fold.
    # This prevents data leakage — the encoder must never see the test data
    # before being evaluated on it.
    ohe.fit(X_train[categorical_cols])

    # Transform both train and test using the SAME fitted encoder
    X_train_cat = ohe.transform(X_train[categorical_cols])
    X_test_cat = ohe.transform(X_test[categorical_cols])

    # Convert the encoded arrays back into DataFrames with proper column names,
    # so we can concatenate them with the numeric columns later
    encoded_col_names = ohe.get_feature_names_out(categorical_cols)
    X_train_cat = pd.DataFrame(X_train_cat, columns=encoded_col_names, index=X_train.index)
    X_test_cat = pd.DataFrame(X_test_cat, columns=encoded_col_names, index=X_test.index)

    # ---- Combine encoded categorical columns with numeric columns ----
    X_train_final = pd.concat([X_train[numeric_cols], X_train_cat], axis=1)
    X_test_final = pd.concat([X_test[numeric_cols], X_test_cat], axis=1)

    # ---- Scale features for models that need it (Linear Regression, KNN) ----
    # Fit the scaler ONLY on training data, again to avoid leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    # ---- Define the 5 models ----
    # Linear Regression and KNN use the SCALED features
    # Tree-based models (Decision Tree, Random Forest) use the UNSCALED features,
    # since trees split on raw values and don't need scaling
    linear_model = LinearRegression()
    knn_model = KNeighborsRegressor()
    tree_model = DecisionTreeRegressor(random_state=42)
    rf_model = RandomForestRegressor(random_state=42)

    # Stacking combines the base models above, using a Linear Regression
    # as the final "meta-model" that learns how to best combine their predictions
    stacking_model = StackingRegressor(
        estimators=[
            ("linear", LinearRegression()),
            ("knn", KNeighborsRegressor()),
            ("tree", DecisionTreeRegressor(random_state=42)),
            ("rf", RandomForestRegressor(random_state=42,n_estimators=50))
        ],
        final_estimator=LinearRegression()
        ,cv=3
    )

    # ---- Train and evaluate each model ----

    # Linear Regression (scaled features)
    linear_model.fit(X_train_scaled, y_train)
    linear_preds = linear_model.predict(X_test_scaled)
    results["Linear Regression"]["rmse"].append(root_mean_squared_error(y_test, linear_preds))
    results["Linear Regression"]["mae"].append(mean_absolute_error(y_test, linear_preds))
    results["Linear Regression"]["r2"].append(r2_score(y_test, linear_preds))

    # KNN (scaled features)
    knn_model.fit(X_train_scaled, y_train)
    knn_preds = knn_model.predict(X_test_scaled)
    results["KNN"]["rmse"].append(root_mean_squared_error(y_test, knn_preds))
    results["KNN"]["mae"].append(mean_absolute_error(y_test, knn_preds))
    results["KNN"]["r2"].append(r2_score(y_test, knn_preds))

    # Decision Tree (unscaled features)
    tree_model.fit(X_train_final, y_train)
    tree_preds = tree_model.predict(X_test_final)
    results["Decision Tree"]["rmse"].append(root_mean_squared_error(y_test, tree_preds))
    results["Decision Tree"]["mae"].append(mean_absolute_error(y_test, tree_preds))
    results["Decision Tree"]["r2"].append(r2_score(y_test, tree_preds))

    # Random Forest (unscaled features)
    rf_model.fit(X_train_final, y_train)
    rf_preds = rf_model.predict(X_test_final)
    results["Random Forest"]["rmse"].append(root_mean_squared_error(y_test, rf_preds))
    results["Random Forest"]["mae"].append(mean_absolute_error(y_test, rf_preds))
    results["Random Forest"]["r2"].append(r2_score(y_test, rf_preds))
    # Stacking (unscaled features — base learners here don't strictly need scaling,
    # since only one of the four base models, Linear Regression, is scale-sensitive;
    # we'll revisit this choice if results look off)
    stacking_model.fit(X_train_final, y_train)
    stacking_preds = stacking_model.predict(X_test_final)
    results["Stacking"]["rmse"].append(root_mean_squared_error(y_test, stacking_preds,))
    results["Stacking"]["mae"].append(mean_absolute_error(y_test, stacking_preds))
    results["Stacking"]["r2"].append(r2_score(y_test, stacking_preds))

    # Print progress so we can see the loop working fold by fold
    print(f"Fold {fold_num}/5 complete")

print("\nAll folds complete.")

Fold 1/5 complete
Fold 2/5 complete
Fold 3/5 complete
Fold 4/5 complete
Fold 5/5 complete
Fold 6/5 complete
Fold 7/5 complete
Fold 8/5 complete
Fold 9/5 complete
Fold 10/5 complete

All folds complete.


In [25]:
# Flatten results into a long-format DataFrame: one row per model per fold per metric
rows = []
for model_name, metrics in results.items():
    for fold_i in range(10):
        rows.append({
            "encoder": "onehot",
            "model": model_name,
            "fold": fold_i + 1,
            "rmse": metrics["rmse"][fold_i],
            "mae": metrics["mae"][fold_i],
            "r2": metrics["r2"][fold_i]
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("onehot_scores.csv", index=False)

print("Saved results to ../results/onehot_scores.csv")

Saved results to ../results/onehot_scores.csv
